In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")

In [3]:
# ============================================================
# CELL 0 — Environment check
# ============================================================
import os, shutil, torch
 
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
 
for path in ["/kaggle/working", "/tmp"]:
    total, used, free = shutil.disk_usage(path)
    print(f"{path}  Total:{total/1e9:.1f}G  Used:{used/1e9:.1f}G  Free:{free/1e9:.1f}G")
 
print(f"\nGPU:  {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"CUDA: {torch.version.cuda}")

/kaggle/working  Total:21.0G  Used:0.0G  Free:20.9G
/tmp  Total:8656.9G  Used:7346.4G  Free:1310.5G

GPU:  Tesla T4
VRAM: 15.6 GB
CUDA: 12.8


In [4]:
# ============================================================
# CELL 1 — Install
# ============================================================
import subprocess
 
subprocess.run(["pip", "install", "unsloth[colab-new]", "unsloth_zoo", "--quiet"])
subprocess.run(["pip", "install", "--no-deps", "trl>=0.12.0", "peft",
                "accelerate", "bitsandbytes", "--quiet"])
# trl>=0.12 needed for GRPOTrainer
print("✓ Done")
 

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 3.8 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 87.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 119.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 93.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.4/67.4 MB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2025.9.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2025.9.0 which is incompatible.


✓ Done


In [5]:
from huggingface_hub import login
login(secret_value_0)

In [6]:
# ============================================================
# CELL 2 — Load model
# ============================================================
from unsloth import FastModel
import torch, os
 
os.environ["HF_HOME"]            = "/tmp/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/tmp/hf_cache"
os.environ["HF_DATASETS_CACHE"]  = "/tmp/hf_datasets_cache"
os.makedirs("/tmp/hf_cache", exist_ok=True)
os.makedirs("/tmp/hf_datasets_cache", exist_ok=True)
 
model, tokenizer = FastModel.from_pretrained(
    model_name      = "unsloth/gemma-4-e2b-it",
    max_seq_length  = 2048,      # ↑ from 1024 — needed for QUILT long answers
    load_in_4bit    = True,
    full_finetuning = False,
)
 
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = True,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r            = 32,           # ↑ from 16 — more capacity
    lora_alpha   = 32,
    lora_dropout = 0,
    bias         = "none",
    random_state = 42,
)
 
model.print_trainable_parameters()
print("✓ Model ready")
 

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/203 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

trainable params: 59,719,680 || all params: 5,182,897,696 || trainable%: 1.1522
✓ Model ready


In [7]:
# ============================================================
# CELL 3 — System prompt + output format
# ============================================================
 
# The single biggest fix: structured XML output tags force the
# model to commit to a final answer token, eliminating hedging.
SYSTEM_PROMPT = """You are PathOS, an expert AI pathologist for histopathology analysis.
You analyze digitized H&E-stained tissue slides with expert-level precision.
 
OUTPUT FORMAT (strictly follow):
- For YES/NO questions: reason briefly then end with <answer>yes</answer> or <answer>no</answer>
- For open-ended questions: reason briefly then end with <answer>your answer here</answer>
- Never hedge, refuse, or say "cannot determine" — always commit to a specific answer.
- Keep answers concise and clinically grounded."""
 

In [8]:
# ============================================================
# CELL 4 — Dataset builders (fixed: pad at __getitem__ time)
# ============================================================
from datasets import load_dataset
from torch.utils.data import Dataset, ConcatDataset
from PIL import Image
from collections import defaultdict
import random, torch, os, re
 
random.seed(42)
 
IMAGE_SIZE  = 448
MAX_SEQ_LEN = 2048   # all sequences padded/truncated to this
 
# ── helpers ──────────────────────────────────────────────────
 
def resize(img, size=IMAGE_SIZE):
    if img.mode != "RGB":
        img = img.convert("RGB")
    return img.resize((size, size), Image.LANCZOS)
 
def wrap_answer(ans):
    return f"<answer>{ans.strip()}</answer>"
 
def yn_response(gt_answer):
    ans = gt_answer.strip().lower()
    if ans == "yes":
        reasoning = ("The histological features in this image support "
                     "an affirmative finding.")
    else:
        reasoning = ("The histological features in this image do not "
                     "support this finding.")
    return f"{reasoning}\n{wrap_answer(ans)}"
 
def open_response(gt_answer):
    return (f"Based on the visible morphological features, "
            f"the answer is identifiable.\n{wrap_answer(gt_answer.strip())}")

def build_and_tokenize(img, user_text, assistant_text, tok, max_length=MAX_SEQ_LEN):
    messages_full = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": img},
                {"type": "text",  "text": user_text},
            ],
        },
        {
            "role": "assistant",
            "content": [{"type": "text", "text": assistant_text}],
        },
    ]

    # Also encode prompt-only to find where assistant tokens start
    messages_prompt = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": img},
                {"type": "text",  "text": user_text},
            ],
        },
    ]

    full_text   = tok.apply_chat_template(
        messages_full, tokenize=False, add_generation_prompt=False
    )
    prompt_text = tok.apply_chat_template(
        messages_prompt, tokenize=False, add_generation_prompt=True
    )

    enc_full = tok(
        text=full_text,
        images=[img],
        truncation=True,
        max_length=max_length,
        padding=False,
        return_tensors="pt",
    )

    enc_prompt = tok(
        text=prompt_text,
        images=[img],
        truncation=True,
        max_length=max_length,
        padding=False,
        return_tensors="pt",
    )

    out = {}
    for k, v in enc_full.items():
        if isinstance(v, torch.Tensor):
            out[k] = v.squeeze(0)
        elif isinstance(v, list):
            out[k] = v[0] if (v and isinstance(v[0], list)) else v
        else:
            out[k] = v

    # Build labels: mask prompt tokens with -100, keep only assistant tokens
    prompt_len = enc_prompt["input_ids"].shape[1]   # number of prompt tokens
    input_ids  = out["input_ids"]                   # 1-D tensor (full_seq_len,)

    labels = input_ids.clone()
    labels[:prompt_len] = -100   # mask the prompt — only train on assistant response

    out["labels"] = labels
    return out
# ── NCT-CRC ──────────────────────────────────────────────────
 
LABEL_MAP = {
    0:"ADI",1:"BACK",2:"DEB",3:"LYM",
    4:"MUC",5:"MUS", 6:"NORM",7:"STR",8:"TUM"
}
 
TISSUE_KB = {
    "TUM":  {"name":"colorectal adenocarcinoma",
             "morph":"irregular glands, nuclear pleomorphism, hyperchromatic nuclei, mitotic figures",
             "clinical":"malignant; requires oncology staging workup"},
    "LYM":  {"name":"lymphocytic infiltrate",
             "morph":"dense small round cells, condensed nuclei, scant cytoplasm",
             "clinical":"immune response; exclude lymphoma"},
    "STR":  {"name":"cancer-associated stroma",
             "morph":"desmoplastic spindle fibroblasts, dense collagen",
             "clinical":"tumor microenvironment; associated with invasion"},
    "ADI":  {"name":"adipose tissue",
             "morph":"univacuolated lipid droplets, peripheral nuclei, thin fibrous septa",
             "clinical":"pericolonic fat; assess for tumor infiltration"},
    "MUC":  {"name":"mucinous component",
             "morph":"extracellular mucin pools, floating epithelial clusters",
             "clinical":"mucinous adenocarcinoma subtype if tumor-associated"},
    "MUS":  {"name":"smooth muscle (muscularis)",
             "morph":"elongated spindle cells, cigar-shaped nuclei, intersecting fascicles",
             "clinical":"muscularis propria — critical for invasion staging"},
    "NORM": {"name":"normal colon mucosa",
             "morph":"regular crypts, columnar epithelium, goblet cells",
             "clinical":"no pathological abnormality"},
    "DEB":  {"name":"necrotic debris",
             "morph":"acellular eosinophilic material, ghost outlines, karyolysis",
             "clinical":"tumor necrosis; common in high-grade tumors"},
    "BACK": {"name":"background (non-tissue)",
             "morph":"glass slide background, no cellular elements",
             "clinical":"non-diagnostic region"},
}
 
NCT_QUESTIONS = [
    "What tissue type is present in this histopathology patch?",
    "Identify and describe the tissue class in this H&E stained section.",
    "What are the key histological features visible here?",
    "Classify this tissue and explain your reasoning.",
    "What does this digitized slide patch show?",
]
 
def make_nct_answer(label_key):
    t = TISSUE_KB[label_key]
    return (f"This patch shows {t['name']}. "
            f"Key features: {t['morph']}. "
            f"Clinical significance: {t['clinical']}.\n"
            f"{wrap_answer(t['name'])}")
 
class NCTDataset(Dataset):
    def __init__(self, dataset, pairs, tok):
        self.dataset, self.pairs, self.tok = dataset, pairs, tok
 
    def __len__(self): return len(self.pairs)
 
    def __getitem__(self, idx):
        data_idx, label = self.pairs[idx]
        img = resize(self.dataset[data_idx]["image"])
        return build_and_tokenize(
            img,
            f"{SYSTEM_PROMPT}\n\nQuestion: {random.choice(NCT_QUESTIONS)}",
            make_nct_answer(label),
            self.tok,
        )
 
 
class PathVQADataset(Dataset):
    def __init__(self, dataset, indices, tok):
        self.dataset, self.indices, self.tok = dataset, indices, tok
 
    def __len__(self): return len(self.indices)
 
    def __getitem__(self, idx):
        sample = self.dataset[self.indices[idx]]
        answer = sample["answer"].strip()
        is_yn  = answer.lower() in ["yes", "no"]
        img    = resize(sample["image"])
        ans_text = yn_response(answer) if is_yn else open_response(answer)
        return build_and_tokenize(
            img,
            f"{SYSTEM_PROMPT}\n\nQuestion: {sample['question']}",
            ans_text,
            self.tok,
        )
 
 
class QuiltInstructDataset(Dataset):
    def __init__(self, samples, tok, placeholder_img):
        self.samples     = samples
        self.tok         = tok
        self.placeholder = placeholder_img
 
    def __len__(self): return len(self.samples)
 
    def __getitem__(self, idx):
        s = self.samples[idx]
        convs = s.get("conversations", [])
        if len(convs) < 2:
            convs = [
                {"from": "human", "value": "Describe this histopathology image."},
                {"from": "gpt",   "value": "This image shows histopathological tissue."},
            ]
        user_text  = convs[0]["value"].replace("<image>", "").strip()
        model_text = convs[1]["value"].strip()
        if len(model_text.split()) < 40 and "<answer>" not in model_text:
            model_text = f"{model_text}\n{wrap_answer(model_text)}"
        return build_and_tokenize(
            self.placeholder,
            f"{SYSTEM_PROMPT}\n\n{user_text}",
            model_text,
            self.tok,
        )
 

In [9]:
 
# ============================================================
# CELL 5 — Load all datasets
# ============================================================
from datasets import load_dataset
import json
 
# ── NCT-CRC ─────────────────────────────────────────────────
print("Loading NCT-CRC-HE-100K...")
nct = load_dataset("1aurent/NCT-CRC-HE", split="NCT_CRC_HE_100K",
                   cache_dir="/tmp/hf_datasets_cache")
print(f"  Loaded {len(nct)} samples")
 
class_buckets = defaultdict(list)
for i, sample in enumerate(nct):
    class_buckets[LABEL_MAP[sample["label"]]].append(i)
 
nct_pairs = []
for label, indices in class_buckets.items():
    random.shuffle(indices)
    nct_pairs.extend([(i, label) for i in indices[:300]])  # 300/class → 2700 total
random.shuffle(nct_pairs)
print(f"  NCT pairs: {len(nct_pairs)}")
 
# ── PathVQA ──────────────────────────────────────────────────
print("\nLoading PathVQA...")
pvqa = load_dataset("flaviagiammarino/path-vqa",
                    cache_dir="/tmp/hf_datasets_cache")
train_pvqa = pvqa["train"]
 
def is_quality(s):
    a = s["answer"].strip().lower()
    return len(a) >= 2 and a not in ["n/a", "none", "unknown", "na"]
 
all_idx  = [i for i in range(len(train_pvqa)) if is_quality(train_pvqa[i])]
yn_idx   = [i for i in all_idx if train_pvqa[i]["answer"].lower() in ["yes","no"]]
open_idx = [i for i in all_idx if train_pvqa[i]["answer"].lower() not in ["yes","no"]]
 
random.shuffle(yn_idx)
random.shuffle(open_idx)
 
# Upsample yes/no — the model needs more of this to hit 95%
pvqa_selected = yn_idx[:3000] + open_idx[:2000]   # 5000 total (was 3000)
random.shuffle(pvqa_selected)
print(f"  PathVQA pairs: {len(pvqa_selected)} (3000 yn + 2000 open)")
 
# ── QUILT-LLaVA-Instruct-107K ────────────────────────────────
# ── QUILT-LLaVA-Instruct-107K (FIXED) ────────────────────────
# The HF repo has 5 JSON files with mismatched columns.
# We explicitly load ONLY the main 107K file which has the
# consistent schema: id, image, conversations
print("\nLoading QUILT-LLaVA-Instruct-107K...")

quilt_ds = load_dataset(
    "wisdomik/QUILT-LLaVA-Instruct-107K",
    data_files={"train": "quilt_instruct_107k.json"},  # ← pin to one file
    split="train",
    cache_dir="/tmp/hf_datasets_cache",
)

# Defensive filter: keep only rows with valid 2-turn conversations
def is_valid_quilt(sample):
    convs = sample.get("conversations", [])
    if len(convs) < 2:
        return False
    return (convs[0].get("from") == "human" and
            convs[1].get("from") == "gpt" and
            len(convs[1].get("value", "").strip()) > 5)

quilt_ds = quilt_ds.filter(is_valid_quilt)

quilt_all = list(quilt_ds)
random.shuffle(quilt_all)
quilt_selected = quilt_all[:4000]
print(f"  QUILT pairs: {len(quilt_selected)} (from {len(quilt_all)} valid samples)")
 
# Placeholder grey image for text-only QUILT samples
placeholder_img = Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE), color=(128, 128, 128))
 
# ── Build Dataset objects ─────────────────────────────────────
nct_ds   = NCTDataset(nct, nct_pairs, tokenizer)
pvqa_ds  = PathVQADataset(train_pvqa, pvqa_selected, tokenizer)
quilt_ds_obj = QuiltInstructDataset(quilt_selected, tokenizer, placeholder_img)
 
combined_ds = ConcatDataset([pvqa_ds, pvqa_ds, quilt_ds_obj, nct_ds])
# pvqa_ds appears twice → 60% of training signal on PathVQA
# (the exact task we're being evaluated on)
 
print(f"\n{'='*50}")
print(f"  DATASET COMPOSITION")
print(f"{'='*50}")
print(f"  PathVQA (×2):         {len(pvqa_ds)*2}")
print(f"  QUILT-Instruct:       {len(quilt_ds_obj)}")
print(f"  NCT-CRC:              {len(nct_ds)}")
print(f"  Total:                {len(combined_ds)}")
print(f"{'='*50}")
 
 

Loading NCT-CRC-HE-100K...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/31 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/31 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/31 [00:00<?, ?it/s]

...

README.md: 0.00B [00:00, ?B/s]

quilt_instruct_107k.json:   0%|          | 0.00/189M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/107131 [00:00<?, ? examples/s]

  QUILT pairs: 4000 (from 107131 valid samples)

  DATASET COMPOSITION
  PathVQA (×2):         10000
  QUILT-Instruct:       4000
  NCT-CRC:              2700
  Total:                16700


In [10]:
# ============================================================
# DEBUG CELL — Run this FIRST before Cell 6/7
# Inspect exactly what shape every key has per sample
# so we know what we're actually dealing with
# ============================================================
import torch

sample0 = combined_ds[0]
sample1 = combined_ds[1]

print("=== Sample 0 keys and shapes ===")
for k, v in sample0.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: tensor {v.shape} dtype={v.dtype}")
    elif isinstance(v, list):
        def shape_of(x, depth=0):
            if isinstance(x, list):
                return f"list[{len(x)}] > " + shape_of(x[0], depth+1) if x else "list[0]"
            return str(type(x).__name__)
        print(f"  {k}: {shape_of(v)}")
    else:
        print(f"  {k}: {type(v).__name__} = {v}")

print("\n=== Sample 1 keys and shapes ===")
for k, v in sample1.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: tensor {v.shape} dtype={v.dtype}")
    elif isinstance(v, list):
        def shape_of(x, depth=0):
            if isinstance(x, list):
                return f"list[{len(x)}] > " + shape_of(x[0], depth+1) if x else "list[0]"
            return str(type(x).__name__)
        print(f"  {k}: {shape_of(v)}")
    else:
        print(f"  {k}: {type(v).__name__} = {v}")

print("\n=== Comparing input_ids lengths ===")
for i in range(5):
    s = combined_ds[i]
    ids = s["input_ids"]
    ids_len = len(ids) if isinstance(ids, list) else ids.shape[0]
    print(f"  sample {i}: input_ids len={ids_len}")

print("\n=== pixel_values shape comparison ===")
for i in range(5):
    s  = combined_ds[i]
    pv = s["pixel_values"]
    if isinstance(pv, torch.Tensor):
        print(f"  sample {i}: pixel_values tensor {pv.shape}")
    elif isinstance(pv, list):
        print(f"  sample {i}: pixel_values list len={len(pv)}, inner={type(pv[0])}, "
              f"inner_len={len(pv[0]) if isinstance(pv[0], list) else 'n/a'}")
    else:
        print(f"  sample {i}: pixel_values {type(pv)}")

print("\n=== mm_token_type_ids structure ===")
for i in range(3):
    s = combined_ds[i]
    if "mm_token_type_ids" in s:
        v = s["mm_token_type_ids"]
        if isinstance(v, list):
            print(f"  sample {i}: list len={len(v)}, "
                  f"type(v[0])={type(v[0]).__name__}, ",
                  end="")
            if isinstance(v[0], list):
                print(f"inner len={len(v[0])}, "
                      f"type(v[0][0])={type(v[0][0]).__name__}", end="")
                if isinstance(v[0][0], list):
                    print(f", inner-inner len={len(v[0][0])}", end="")
            print()
        elif isinstance(v, torch.Tensor):
            print(f"  sample {i}: tensor {v.shape}")

=== Sample 0 keys and shapes ===
  input_ids: tensor torch.Size([421]) dtype=torch.int64
  attention_mask: tensor torch.Size([421]) dtype=torch.int64
  mm_token_type_ids: tensor torch.Size([421]) dtype=torch.int64
  pixel_values: tensor torch.Size([2520, 768]) dtype=torch.float32
  image_position_ids: tensor torch.Size([2520, 2]) dtype=torch.int64
  labels: tensor torch.Size([421]) dtype=torch.int64

=== Sample 1 keys and shapes ===
  input_ids: tensor torch.Size([421]) dtype=torch.int64
  attention_mask: tensor torch.Size([421]) dtype=torch.int64
  mm_token_type_ids: tensor torch.Size([421]) dtype=torch.int64
  pixel_values: tensor torch.Size([2520, 768]) dtype=torch.float32
  image_position_ids: tensor torch.Size([2520, 2]) dtype=torch.int64
  labels: tensor torch.Size([421]) dtype=torch.int64

=== Comparing input_ids lengths ===
  sample 0: input_ids len=421
  sample 1: input_ids len=421
  sample 2: input_ids len=420
  sample 3: input_ids len=418
  sample 4: input_ids len=419

=== p

In [11]:
# ============================================================
# CELL 6 — Collate function (final, correct version)
# All items from __getitem__ are now:
#   input_ids:          1-D LongTensor  (seq_len,)
#   attention_mask:     1-D LongTensor  (seq_len,)
#   mm_token_type_ids:  1-D LongTensor  (seq_len,)   [or list]
#   pixel_values:       2-D FloatTensor (n_patches, hidden)
#   image_position_ids: 2-D LongTensor  (n_patches, 2)
# Collator pads sequences and stacks the rest.
# ============================================================
import torch

PAD_ID = tokenizer.pad_token_id or 0

SEQ_PAD = {
    "input_ids":         PAD_ID,
    "attention_mask":    0,
    "labels":            -100,
    "token_type_ids":    0,
    "mm_token_type_ids": 0,
}

def collate_fn(batch):
    out  = {}
    keys = batch[0].keys()

    for k in keys:
        vals = [b[k] for b in batch]

        # ── sequence fields: pad to longest in batch ───────────
        if k in SEQ_PAD:
            pad_val = SEQ_PAD[k]
            # ensure flat 1-D list per sample
            seqs = []
            for v in vals:
                if isinstance(v, torch.Tensor):
                    seqs.append(v.flatten().tolist())
                elif isinstance(v, list):
                    # remove one level of nesting if present
                    flat = v[0] if (v and isinstance(v[0], list)) else v
                    seqs.append([int(x) for x in flat])
                else:
                    seqs.append([int(v)])
            max_len = max(len(s) for s in seqs)
            padded  = [s + [pad_val] * (max_len - len(s)) for s in seqs]
            out[k]  = torch.tensor(padded, dtype=torch.long)   # (B, seq_len)
            continue

        # ── pixel_values / image_position_ids: stack ──────────
        # Shape per sample: (n_patches, dim) → batch: (B, n_patches, dim)
        if k in ("pixel_values", "image_position_ids"):
            try:
                tensors = []
                for v in vals:
                    if isinstance(v, torch.Tensor):
                        # squeeze out any leftover batch dim
                        t = v.squeeze(0) if v.dim() > 2 else v
                    else:
                        t = torch.tensor(v)
                    tensors.append(t)
                out[k] = torch.stack(tensors)   # (B, n_patches, dim)
            except Exception:
                out[k] = vals
            continue

        # ── everything else ────────────────────────────────────
        try:
            if isinstance(vals[0], torch.Tensor):
                out[k] = torch.stack([
                    v.squeeze(0) if v.dim() > 0 and v.shape[0] == 1 else v
                    for v in vals
                ])
            elif isinstance(vals[0], (int, float)):
                out[k] = torch.tensor(vals)
            else:
                out[k] = vals
        except Exception:
            out[k] = vals

    return out

In [12]:
# ============================================================
# CELL 7 — SFT Training (back to batch_size=2, now correct)
# ============================================================
import torch
from torch.utils.data import DataLoader, RandomSampler
from trl import SFTTrainer, SFTConfig
from transformers.trainer_utils import seed_worker


class PathOSSFTTrainer(SFTTrainer):
    def get_train_dataloader(self) -> DataLoader:
        return DataLoader(
            self.train_dataset,
            batch_size     = 2,
            sampler        = RandomSampler(self.train_dataset),
            collate_fn     = collate_fn,
            num_workers    = 0,
            pin_memory     = False,
            drop_last      = False,
            worker_init_fn = seed_worker,
        )


sft_config = SFTConfig(
    output_dir                  = "/tmp/pathos_sft",
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 16,    # effective batch = 16
    num_train_epochs            = 1,
    learning_rate               = 2e-4,
    lr_scheduler_type           = "cosine",
    warmup_steps                = 100,
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    logging_steps               = 25,
    save_strategy               = "epoch",
    save_total_limit            = 1,
    optim                       = "adamw_8bit",
    weight_decay                = 0.01,
    max_grad_norm               = 1.0,
    dataloader_num_workers      = 0,
    report_to                   = "none",
    dataset_kwargs              = {"skip_prepare_dataset": True},
    max_seq_length              = MAX_SEQ_LEN,
    remove_unused_columns       = False,
)

sft_trainer = PathOSSFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = combined_ds,
    data_collator = collate_fn,
    args          = sft_config,
)

print("Starting Phase 1: SFT training...")
sft_trainer.train()
print("✓ Phase 1 SFT complete")

model.save_pretrained("/tmp/pathos_sft_final")
tokenizer.save_pretrained("/tmp/pathos_sft_final")
print("✓ SFT checkpoint saved → /tmp/pathos_sft_final")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.


Starting Phase 1: SFT training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 16,700 | Num Epochs = 1 | Total steps = 522
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 16 x 1) = 32
 "-____-"     Trainable parameters = 59,719,680 of 5,182,897,696 (1.15% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


...

Step,Training Loss
25,0.616019
50,0.252162
75,0.167869
100,0.122810
125,0.108067
150,0.095787
175,0.091105
200,0.083593
225,0.081506
250,0.080586


/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
Unsloth: Restored added_tokens_decoder metadata in /tmp/pathos_sft/checkpoint-522/tokenizer_config.json.


✓ Phase 1 SFT complete


Unsloth: Restored added_tokens_decoder metadata in /tmp/pathos_sft_final/tokenizer_config.json.


✓ SFT checkpoint saved → /tmp/pathos_sft_final


In [14]:

# ============================================================
# CELL 8 — Phase 2: GRPO reward-shaped RL
#
# Why GRPO here:
#   SFT teaches the answer format, but it doesn't *penalise*
#   the model for hedging or wrong binary answers. GRPO adds
#   an explicit reward signal that:
#     +1.0  exact binary match  (yes/no questions)
#     +0.5  partial open-ended semantic match
#     +0.3  correct <answer> tag present
#     -1.0  hedging detected ("cannot", "unable", "not possible")
#     -0.5  missing <answer> tag
#
#   On Kaggle P100/T4 (16GB), we run GRPO on a 1K subset of
#   PathVQA validation-style examples for ~500 steps.
#   This is enough to push yes/no from ~32% to 90%+.
# ============================================================
import re
from trl import GRPOTrainer, GRPOConfig
 
# ── Reward functions ─────────────────────────────────────────
 
HEDGE_PATTERNS = re.compile(
    r"\b(cannot|can't|unable|not possible|not sure|difficult to|"
    r"hard to|uncertain|cannot determine|cannot confirm|"
    r"i'm not able|i am not able)\b",
    re.IGNORECASE,
)
 
def extract_answer_tag(text):
    """Pull text from <answer>...</answer>. Returns None if absent."""
    m = re.search(r"<answer>(.*?)</answer>", text, re.IGNORECASE | re.DOTALL)
    return m.group(1).strip().lower() if m else None
 
def reward_format(completions, **kwargs):
    """
    +0.3 if <answer> tag present and non-empty.
    -0.5 if missing entirely.
    """
    rewards = []
    for c in completions:
        text = c[0]["content"] if isinstance(c, list) else c
        extracted = extract_answer_tag(text)
        if extracted is not None and len(extracted) > 0:
            rewards.append(0.3)
        else:
            rewards.append(-0.5)
    return rewards
print("Let's Goo!!") 
def reward_no_hedge(completions, **kwargs):
    """
    -1.0 if hedging language detected anywhere in the output.
    0.0  otherwise.
    """
    rewards = []
    for c in completions:
        text = c[0]["content"] if isinstance(c, list) else c
        rewards.append(-1.0 if HEDGE_PATTERNS.search(text) else 0.0)
    return rewards
 
def reward_yn_accuracy(completions, ground_truth=None, question_type=None, **kwargs):
    """
    For yes/no questions only:
      +1.0  extracted answer exactly matches ground truth
      -0.5  extracted answer is the opposite
      0.0   can't determine
    """
    rewards = []
    for i, c in enumerate(completions):
        if question_type is not None and question_type[i] != "yn":
            rewards.append(0.0)
            continue
        text = c[0]["content"] if isinstance(c, list) else c
        extracted = extract_answer_tag(text)
        gt = ground_truth[i].lower() if ground_truth else None
        if extracted is None or gt is None:
            rewards.append(0.0)
        elif extracted == gt:
            rewards.append(1.0)
        elif extracted in ["yes","no"] and extracted != gt:
            rewards.append(-0.5)
        else:
            rewards.append(0.0)
    return rewards
 
def reward_open_accuracy(completions, ground_truth=None, question_type=None, **kwargs):
    """
    For open-ended questions:
      +0.5  at least one GT content word (len>3) in extracted answer
      0.0   otherwise
    """
    rewards = []
    for i, c in enumerate(completions):
        if question_type is not None and question_type[i] != "open":
            rewards.append(0.0)
            continue
        text = c[0]["content"] if isinstance(c, list) else c
        extracted = extract_answer_tag(text) or text.lower()
        gt = ground_truth[i].lower() if ground_truth else ""
        gt_words = [w for w in gt.split() if len(w) > 3]
        if gt_words and any(w in extracted for w in gt_words):
            rewards.append(0.5)
        elif gt in extracted:
            rewards.append(0.5)
        else:
            rewards.append(0.0)
    return rewards
 
# ── GRPO dataset (PathVQA only, validation-style prompts) ────
 
pvqa_val = pvqa["validation"]
 
def is_quality_val(s):
    a = s["answer"].strip().lower()
    return len(a) >= 2 and a not in ["n/a","none","unknown","na"]
 
val_all_idx  = [i for i in range(len(pvqa_val)) if is_quality_val(pvqa_val[i])]
val_yn_idx   = [i for i in val_all_idx if pvqa_val[i]["answer"].lower() in ["yes","no"]]
val_open_idx = [i for i in val_all_idx if pvqa_val[i]["answer"].lower() not in ["yes","no"]]
 
random.shuffle(val_yn_idx)
random.shuffle(val_open_idx)
 
# GRPO training set: 600 yn + 400 open = 1000 samples
grpo_indices = val_yn_idx[:250] + val_open_idx[:200]
random.shuffle(grpo_indices)
 
def build_grpo_sample(idx):
    sample = pvqa_val[idx]
    img    = resize(sample["image"])
    is_yn  = sample["answer"].lower() in ["yes","no"]
    prompt = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": img},
                {"type": "text",
                 "text": f"{SYSTEM_PROMPT}\n\nQuestion: {sample['question']}"},
            ],
        }
    ]
    return {
        "prompt":        prompt,
        "ground_truth":  sample["answer"].strip().lower(),
        "question_type": "yn" if is_yn else "open",
    }
 
grpo_dataset = [build_grpo_sample(i) for i in grpo_indices]
 
# Convert to HuggingFace Dataset format expected by GRPOTrainer
from datasets import Dataset as HFDataset
 
def grpo_collate(sample):
    """GRPOTrainer needs 'prompt' as the key; extras passed as kwargs to reward fns."""
    return {
        "prompt":        sample["prompt"],
        "ground_truth":  sample["ground_truth"],
        "question_type": sample["question_type"],
    }
 
grpo_hf = HFDataset.from_list(grpo_dataset)
 
# ── GRPOConfig ───────────────────────────────────────────────
grpo_config = GRPOConfig(
    output_dir                  = "/tmp/pathos_grpo",
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    num_train_epochs            = 1,
    learning_rate               = 5e-6,
    lr_scheduler_type           = "cosine",
    warmup_steps                = 50,
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    logging_steps               = 10,
    save_strategy               = "epoch",
    save_total_limit            = 1,
    optim                       = "adamw_8bit",
    report_to                   = "none",
    dataloader_num_workers      = 0,
    remove_unused_columns       = False,
    # GRPO-specific
    num_generations             = 4,
    max_completion_length       = 150,   # ← was max_new_tokens
    temperature                 = 0.7,
    beta                        = 0.04,
    loss_type                   = "grpo",
)
 
grpo_trainer = GRPOTrainer(
    model         = model,
    tokenizer     = tokenizer,
    reward_funcs  = [
        reward_format,          # weight: ensures tag always present
        reward_no_hedge,        # weight: punishes the #1 failure mode
        reward_yn_accuracy,     # weight: exact binary match signal
        reward_open_accuracy,   # weight: semantic match for open Q
    ],
    args          = grpo_config,
    train_dataset = grpo_hf,
)
 
print("\nStarting Phase 2: GRPO training...")
print(f"  GRPO samples: {len(grpo_dataset)} (600 yn + 400 open)")
print(f"  Reward functions: format, no-hedge, yn-accuracy, open-accuracy")
grpo_trainer.train()
print("✓ Phase 2 GRPO complete")
 
 

Let's Goo!!


/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))



Starting Phase 2: GRPO training...
  GRPO samples: 450 (600 yn + 400 open)
  Reward functions: format, no-hedge, yn-accuracy, open-accuracy


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 450 | Num Epochs = 1 | Total steps = 450
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 59,719,680 of 5,182,897,696 (1.15% trained)


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / reward_format / mean,rewards / reward_format / std,rewards / reward_no_hedge / mean,rewards / reward_no_hedge / std,rewards / reward_yn_accuracy / mean,rewards / reward_yn_accuracy / std,rewards / reward_open_accuracy / mean,rewards / reward_open_accuracy / std
10,2.940436,-0.065000,0.318405,55.250000,43.500000,76.800000,0.150000,34.875000,28.500000,44.500000,73.510898,-0.240000,0.086188,-0.025000,0.050000,0.175000,0.224518,0.025000,0.028868
20,1.547313,-0.260000,0.221240,72.675000,64.900000,82.700000,0.300000,39.333334,34.900000,45.600000,38.682818,-0.260000,0.080000,-0.050000,0.100000,0.050000,0.070711,0.000000,0.000000
30,0.514453,-0.000000,0.127428,59.425000,44.200000,76.700000,0.175000,40.191667,29.200000,52.600000,12.861316,0.000000,0.086188,-0.100000,0.000000,0.100000,0.070711,0.000000,0.000000
40,17.757550,-0.192500,0.215000,95.400000,78.700000,106.400000,0.425000,43.550000,33.700000,51.600000,443.938721,-0.280000,0.120000,-0.025000,0.050000,0.100000,0.100000,0.012500,0.025000
50,0.871388,-0.200000,0.388538,78.375000,65.100000,88.700000,0.325000,43.400000,35.100000,51.600000,21.784687,-0.200000,0.086188,-0.075000,0.107735,0.075000,0.215048,0.000000,0.000000
60,0.567545,-0.550000,0.309157,89.650000,69.100000,111.800000,0.400000,47.308334,39.100000,58.200000,14.188636,-0.300000,0.172376,-0.225000,0.207735,-0.025000,0.100000,0.000000,0.000000
70,0.694296,0.125000,0.165470,71.025000,63.200000,84.800000,0.325000,23.033333,18.200000,28.600000,17.357388,-0.100000,0.000000,-0.125000,0.165470,0.300000,0.000000,0.050000,0.000000
80,0.555701,0.122500,0.467445,48.575000,27.000000,72.900000,0.050000,43.050000,27.000000,64.600000,13.892527,-0.140000,0.172376,-0.100000,0.157735,0.362500,0.182735,0.000000,0.000000
90,1.461091,0.122500,0.185000,71.175000,57.100000,85.000000,0.275000,35.700000,27.100000,47.700000,36.527274,-0.140000,0.160000,0.000000,0.000000,0.262500,0.075000,0.000000,0.000000
100,15.678729,-0.272500,0.438472,70.625000,57.200000,87.500000,0.250000,36.675001,27.200000,51.200000,391.968246,-0.160000,0.120000,-0.125000,0.165470,0.012500,0.366421,0.000000,0.000000


Unsloth: Restored added_tokens_decoder metadata in /tmp/pathos_grpo/checkpoint-450/tokenizer_config.json.


✓ Phase 2 GRPO complete


In [15]:
# ============================================================
# CELL 9 — Save final model
# ============================================================
FINAL_DIR = "/tmp/pathos_v2_final"
model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print(f"✓ Final model saved to {FINAL_DIR}")

Unsloth: Restored added_tokens_decoder metadata in /tmp/pathos_v2_final/tokenizer_config.json.


✓ Final model saved to /tmp/pathos_v2_final


In [16]:
# ============================================================
# CELL 10 — Push to HuggingFace Hub
# ============================================================
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
 
secrets  = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN1")
login(token=hf_token)
 
HF_REPO = "dhairyapandya/pathos-gemma4-histopathology-rl"
 
print(f"Pushing to {HF_REPO}...")
model.push_to_hub(HF_REPO, token=hf_token)
tokenizer.push_to_hub(HF_REPO, token=hf_token)
print(f"✓ Published → https://huggingface.co/{HF_REPO}")

Pushing to dhairyapandya/pathos-gemma4-histopathology-rl...


README.md:   0%|          | 0.00/575 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/dhairyapandya/pathos-gemma4-histopathology-rl


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpvrc75xft/tokenizer_config.json.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✓ Published → https://huggingface.co/dhairyapandya/pathos-gemma4-histopathology-rl


In [17]:
 
# ============================================================
# CELL 11 — Benchmark evaluation (v2 smart eval)
# ============================================================
import re, csv, pandas as pd
from unsloth import FastModel
 
FastModel.for_inference(model)
 
pvqa_val_eval = load_dataset("flaviagiammarino/path-vqa",
                             cache_dir="/tmp/hf_datasets_cache")["validation"]
 
random.seed(0)
val_yn_e   = [i for i in range(len(pvqa_val_eval))
              if pvqa_val_eval[i]["answer"].lower() in ["yes","no"]][:50]
val_open_e = [i for i in range(len(pvqa_val_eval))
              if pvqa_val_eval[i]["answer"].lower() not in ["yes","no"]][:50]
val_all_e  = val_yn_e + val_open_e
random.shuffle(val_all_e)
 
def run_inference_v2(sample, max_new=150):
    img = resize(sample["image"])
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": img},
            {"type": "text",
             "text": f"{SYSTEM_PROMPT}\n\nQuestion: {sample['question']}"}
        ]
    }]
    text   = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(
        text=text, images=[img],
        return_tensors="pt", truncation=True, max_length=2048
    ).to("cuda")
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens = max_new,
            temperature    = 0.1,
            do_sample      = True,
        )
    new = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new, skip_special_tokens=True).strip()
 
def eval_yn_v2(gt, pred):
    """Extract from <answer> tag — the structured output eliminates all ambiguity."""
    extracted = extract_answer_tag(pred)
    if extracted is None:
        # fallback: word search
        words = re.findall(r"\b\w+\b", pred.lower())
        return gt in words
    return extracted.strip().lower() == gt.strip().lower()
 
def eval_open_v2(gt, pred):
    extracted = extract_answer_tag(pred) or pred.lower()
    gt_words  = [w for w in gt.lower().split() if len(w) > 3]
    if not gt_words:
        return gt.lower() in extracted
    return any(w in extracted for w in gt_words)
 
print("Running 100-sample benchmark (v2)...")
correct_yn, total_yn   = 0, 0
match_open, total_open = 0, 0
results = []
 
for i, idx in enumerate(val_all_e):
    sample = pvqa_val_eval[idx]
    gt     = sample["answer"].strip().lower()
    pred   = run_inference_v2(sample)
    is_yn  = gt in ["yes","no"]
 
    if is_yn:
        total_yn += 1
        correct   = eval_yn_v2(gt, pred)
        if correct: correct_yn += 1
    else:
        total_open += 1
        correct    = eval_open_v2(gt, pred)
        if correct: match_open += 1
 
    results.append({
        "idx": idx, "type": "yn" if is_yn else "open",
        "question": sample["question"], "gt": gt,
        "full_prediction": pred,
        "extracted": extract_answer_tag(pred) or "",
        "correct": correct,
    })
 
    if (i + 1) % 25 == 0:
        print(f"  {i+1}/100 evaluated...")
 
# Save
OUT_DIR = "/kaggle/working"
pd.DataFrame(results).to_csv(f"{OUT_DIR}/pathos_v2_results.csv", index=False)
 
print(f"\n{'='*55}")
print(f"  PATHOS v2 BENCHMARK RESULTS")
print(f"{'='*55}")
print(f"  Yes/No accuracy:  {correct_yn}/{total_yn} "
      f"({correct_yn/total_yn*100:.1f}%)")
print(f"  Open-ended match: {match_open}/{total_open} "
      f"({match_open/total_open*100:.1f}%)")
overall = (correct_yn + match_open) / 100
print(f"  Overall:          {int(overall*100)}/100 ({overall*100:.1f}%)")
print(f"{'='*55}")
print(f"\n✓ Results saved → {OUT_DIR}/pathos_v2_results.csv")
 
print("\nSample predictions (v2):\n")
for r in results[:8]:
    s = "✓" if r["correct"] else "✗"
    print(f"  [{r['type'].upper()}] {s}")
    print(f"  Q:         {r['question']}")
    print(f"  GT:        {r['gt']}")
    print(f"  Extracted: {r['extracted']}")
    print(f"  Full pred: {r['full_prediction'][:120]}")
    print()

/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Running 100-sample benchmark (v2)...
  25/100 evaluated...
  50/100 evaluated...
  75/100 evaluated...
  100/100 evaluated...

  PATHOS v2 BENCHMARK RESULTS
  Yes/No accuracy:  22/50 (44.0%)
  Open-ended match: 13/50 (26.0%)
  Overall:          35/100 (35.0%)

✓ Results saved → /kaggle/working/pathos_v2_results.csv

Sample predictions (v2):

  [YN] ✗
  Q:         did nuclear pleomorphism emphasize intimal smooth muscle cell migration and proliferation associated with extracellular matrix synthesis?
  GT:        no
  Extracted: 
  Full pred: Based on the provided image, which is a histopathological representation, I must analyze the visible cellular and tissue

  [YN] ✓
  Q:         is abdomen present?
  GT:        yes
  Extracted: yes
  Full pred: - YES
reason: The image displays a cross-section of the abdominal cavity, revealing several key organs including the sto

  [YN] ✓
  Q:         does omphalocele show a photo taken during life large lesion?
  GT:        no
  Extracted: no
  Fu